# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.75594465 -0.86242827  0.592238   -0.32730253 -0.29224895]
 [-0.00743609  0.74994935  0.29038837 -0.0536619  -0.38761236]
 [ 0.42483142 -0.8101959   0.97866046  0.96714006 -0.45507855]
 [ 0.10693065 -0.33143575  0.95193123  0.33155827  0.1115043 ]
 [-0.59611822  0.41735215 -0.31214098  0.62919195 -0.81407983]
 [-0.36781556 -0.22151545 -0.39409079  0.05750017 -0.03233095]
 [-0.90995906  0.79138048 -0.54580982  0.18984568 -0.37711134]
 [ 0.35362079 -0.36997502 -0.96769027 -0.48262173  0.13158198]
 [ 0.68015415 -0.3242628   0.6072383  -0.73690562 -0.75011489]
 [ 0.76455046 -0.43671443 -0.64610819  0.85102377  0.21621518]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a1', 'a1', 'a2', 'a2', 'a2', 'a2', 'a1', 'a2', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 1, 0, 1, 1, 0, 0, 1, 1, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/25 [00:00<?, ?it/s]

SVI:   4%|▍         | 1/25 [00:00<00:23,  1.01it/s]

SVI:   4%|▍         | 1/25 [00:00<00:23,  1.01it/s, loss=2070.2959]

SVI:   8%|▊         | 2/25 [00:00<00:22,  1.01it/s, loss=1891.8186]

SVI:  12%|█▏        | 3/25 [00:00<00:21,  1.01it/s, loss=2087.7314]

...

SVI:  91%|█████████ | 31/34 [00:01<00:03,  1.06s/it, loss=1770.4740]

SVI:  94%|█████████▍| 32/34 [00:01<00:02,  1.06s/it, loss=2050.0024]

SVI:  97%|█████████▋| 33/34 [00:01<00:01,  1.06s/it, loss=2268.9944]

SVI: 100%|██████████| 34/34 [00:01<00:00, 21.16it/s, loss=2268.9944]

SVI: 100%|██████████| 34/34 [00:01<00:00, 21.16it/s, loss=2327.0525]